# Prepare Power Plant Retirement Output

## Data Requirements

This notebook relies on power plant siting data outputs from the GODEEEP experiment. Please extract the downloaded data inside a folder called `power_plant_siting_data` in the `data/input_data` directory of this repository as the paths in this notebook are set to that expectation.

**Dataset Title:** Power Plant Siting Results for GODEEEP

**Description from source:** This dataset contains power plant infrastructure siting information modeled by the Capacity Expansion Regional Feasibility (CERF) model for the GODEEEP project for the purpose of studying power plant landscape evolution under alternative capacity expansion scenarios. 

**Download Link**: https://doi.org/10.5281/zenodo.13695663 

**Reference:**
> Mongird, K., & Thurber, T. (2024). Power Plant Siting Results for GODEEEP (1.0) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.13695663 


### Imports

In [1]:
import pandas as pd
from shapely import Point
import geopandas as gpd
import numpy as np
import os

### Data Paths

In [2]:
# data dir
data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data', 'input_data')

# High Renewables cerf siting results
hr_1km_path = os.path.join(data_dir,'power_plant_siting_data', 'power_plant_additions_1km_net_zero.csv')

# business-as-usual cerf siting results
bau_1km_path = os.path.join(data_dir, 'power_plant_siting_data',  'power_plant_additions_1km_bau.csv')

# High Renewables power plants
hr_plants_path = os.path.join(data_dir, 'power_plant_siting_data',  'power_plants_net_zero.csv')

# business-as-usual power plants
bau_plants_path = os.path.join(data_dir, 'power_plant_siting_data',  'power_plants_bau.csv')

# create infrastructure siting output dir
infrastucture_output_dir = os.path.join(data_dir, 'infrastructure_data_csv')
os.makedirs(infrastucture_output_dir, exist_ok = True)

# wind coord file
wind_coord_file = os.path.join(data_dir, 'wind_coord_data', 'wind_coord.shp')

# output path
retirement_output_path = os.path.join(infrastucture_output_dir, 'retirement_data.csv')

## Process Data

In [3]:
# list of columns to keep in output
col_list = ['scenario', 'region_name', 'technology', 'technology_simple', 
            'cerf_sited', 'sited_year', 'retirement_year', 'unit_size_mw', 
            'xcoord', 'ycoord']

west_states = ['arizona', 'california', 'colorado', 'idaho', 'montana', 'nevada',
               'new_mexico', 'oregon', 'utah','washington', 'wyoming']

#### Get Retired Capacity

##### High Renewables

In [4]:
# read in 1km file to get cerf-sited plants
hr_1km_df = pd.read_csv(hr_1km_path)

# assign that plant is CERF sited
hr_1km_df['cerf_sited'] = 1

# read in plant level file
hr_plants_df = pd.read_csv(hr_plants_path)

# remove CERF sited items, collecting pre-existing
hr_plants_df = hr_plants_df[hr_plants_df.cerf_sited == 0]

# combine the files together to get all operational plants
hr_df = pd.concat([hr_1km_df, hr_plants_df])

# reduce to columns of interest
hr_df = hr_df[col_list]

# get all plants that retired across the experiment timeframe
hr_df_cap_retired = hr_df[hr_df.retirement_year <= 2050]

# calculate the total retired capacity by tech and state
cols = ['scenario', 'region_name', 'technology_simple', 'unit_size_mw']
hr_df_cap_retired = hr_df_cap_retired[cols].groupby(['scenario','region_name', 'technology_simple'], as_index=False).sum()
hr_df_cap_retired = hr_df_cap_retired.rename(columns={'technology_simple':'Technology', 'region_name': 'State'})

# calculate in GW
hr_df_cap_retired['GW'] = hr_df_cap_retired['unit_size_mw']/1000

# reformat data
hr_df_cap_retired = pd.melt(hr_df_cap_retired, id_vars=['Technology', 'State', 'scenario'], value_vars=['GW'], var_name='var', value_name='value')

# set values to negative
hr_df_cap_retired['value'] = hr_df_cap_retired['value']*-1

# west states only
hr_df_cap_retired = hr_df_cap_retired[hr_df_cap_retired.State.isin(west_states)]

##### Business-as-usual

In [5]:
# read in 1km file to get cerf-sited plants
bau_1km_df = pd.read_csv(bau_1km_path)

# assign that plant is CERF sited
bau_1km_df['cerf_sited'] = 1

# read in plant level file
bau_plants_df = pd.read_csv(bau_plants_path)

# remove CERF sited items, collecting pre-existing
bau_plants_df = bau_plants_df[bau_plants_df.cerf_sited == 0]

# combine the files together to get all operational plants
bau_df = pd.concat([bau_1km_df, bau_plants_df])

# reduce to columns of interest
bau_df = bau_df[col_list]

# get all plants that retired across the experiment timeframe
bau_cap_retired = bau_df[bau_df.retirement_year <= 2050]

# calculate the total retired capacity by tech and state
cols = ['scenario', 'region_name', 'technology_simple', 'unit_size_mw']
bau_cap_retired = bau_cap_retired[cols].groupby(['scenario','region_name', 'technology_simple'], as_index=False).sum()
bau_cap_retired = bau_cap_retired.rename(columns={'technology_simple':'Technology', 'region_name': 'State'})

# calculate in GW
bau_cap_retired['GW'] = bau_cap_retired['unit_size_mw']/1000

# reformat data
bau_cap_retired = pd.melt(bau_cap_retired, id_vars=['Technology', 'State', 'scenario'], value_vars=['GW'], var_name='var', value_name='value')

# set values to negative
bau_cap_retired['value'] = bau_cap_retired['value']*-1

# west states only
bau_cap_retired = bau_cap_retired[bau_cap_retired.State.isin(west_states)]

#### Get land area that saw retirements and no new replacements

##### High Renewables

Project-level land use that saw retirement over the experiment timeframe

In [6]:
# CERF-sited plants that retired

# read in 1km file to get cerf-sited plants
hr_1km_df = pd.read_csv(hr_1km_path)

# assign that plant is CERF sited
hr_1km_df['cerf_sited'] = 1

# reduce to retired plants
hr_1km_retired = hr_1km_df[hr_1km_df.retirement_year <= 2050]

# create geometry
geometry = [Point(xy) for xy in zip(hr_1km_retired['xcoord'], hr_1km_retired['ycoord'])]
hr_cerf_retired_gdf = gpd.GeoDataFrame(hr_1km_retired, crs='ESRI:102003', geometry=geometry)
hr_cerf_retired_gdf.set_crs('ESRI:102003', inplace=True)

# Pre-existing plants (non-wind)

# read in plant level file
hr_plants_df = pd.read_csv(hr_plants_path)

# remove CERF sited items, collecting pre-existing only
hr_plants_df = hr_plants_df[hr_plants_df.cerf_sited == 0]

# remove wind facilities
hr_plants_df = hr_plants_df[hr_plants_df.technology_simple != 'Wind']

# reduce to retired plants
hr_plants_retired = hr_plants_df[hr_plants_df.retirement_year <= 2050]
geometry = [Point(xy) for xy in zip(hr_plants_retired['xcoord'], hr_plants_retired['ycoord'])]
hr_non_cerf_retired_gdf = gpd.GeoDataFrame(hr_plants_retired, crs='ESRI:102003', geometry=geometry)
hr_non_cerf_retired_gdf.set_crs('ESRI:102003', inplace=True)

# Pre-existing plants (wind)
wind_coord_df = gpd.read_file(wind_coord_file)
wind_coord_df.set_crs('ESRI:102003', inplace=True)
wind_coord_df = wind_coord_df.rename(columns={'State': 'region_name'})
wind_coord_df['technology_simple'] = 'Wind'

# combine together
hr_retired = pd.concat([hr_cerf_retired_gdf, hr_non_cerf_retired_gdf, wind_coord_df])

# add scenario to missing
hr_retired['scenario'] = 'hr_ira_ccs_climate'

# reduce to columns of interest
hr_retired = hr_retired[['scenario', 'technology_simple', 'region_name', 'geometry']]

# square buffer points to get project level area
hr_retired.geometry  = hr_retired.geometry.buffer(500, cap_style='square')


New infrastruture development

In [7]:
# read in 1km file to get cerf-sited plants
hr_1km_df = pd.read_csv(hr_1km_path)

# create geometry
geometry = [Point(xy) for xy in zip(hr_1km_df['xcoord'], hr_1km_df['ycoord'])]
hr_cerf_sited_gdf = gpd.GeoDataFrame(hr_1km_df, crs='ESRI:102003', geometry=geometry)
hr_cerf_sited_gdf.set_crs('ESRI:102003', inplace=True)

# square buffer points to get project level area
hr_cerf_sited_gdf.geometry  = hr_cerf_sited_gdf.geometry.buffer(500, cap_style='square')

# dissolve to single geometry
hr_cerf_sited_gdf = hr_cerf_sited_gdf.dissolve()

Take the difference to determine land that had retirement but saw no new infrastructure

In [8]:
# the overlap: retired areas where there was new generation
hr_difference = gpd.clip(hr_retired, hr_cerf_sited_gdf)

# remove the overlap from the retired land
hr_retired_open = gpd.overlay(hr_retired, hr_difference, how='difference')

# get land area
hr_retired_open['value'] = hr_retired_open['geometry'].area

# convert to km2
hr_retired_open['value'] = (hr_retired_open['value'] / 1e6) * -1

# rename columns
hr_retired_open = hr_retired_open.rename(columns={'technology_simple':'Technology', 'region_name': 'State'})

# reduce to columns of interest
hr_retired_open = hr_retired_open[['Technology', 'State', 'scenario', 'value']]

# add variable name
hr_retired_open['var'] = 'km-squared'

# sum by state and technology
hr_retired_open = hr_retired_open.groupby(['Technology', 'State', 'scenario', 'var'], as_index=False).sum()

# west states only
hr_retired_open = hr_retired_open[hr_retired_open.State.isin(west_states)]

##### Business-as-usual

Project-level land use that saw retirement over the experiment timeframe

In [9]:
# CERF-sited plants that retired

# read in 1km file to get cerf-sited plants
bau_1km_df = pd.read_csv(bau_1km_path)

# assign that plant is CERF sited
bau_1km_df['cerf_sited'] = 1

# reduce to retired plants
bau_1km_retired = bau_1km_df[bau_1km_df.retirement_year <= 2050]

# create geometry
geometry = [Point(xy) for xy in zip(bau_1km_retired['xcoord'], bau_1km_retired['ycoord'])]
bau_cerf_retired_gdf = gpd.GeoDataFrame(bau_1km_retired, crs='ESRI:102003', geometry=geometry)
bau_cerf_retired_gdf.set_crs('ESRI:102003', inplace=True)

# Pre-existing plants (non-wind)

# read in plant level file
bau_plants_df = pd.read_csv(bau_plants_path)

# remove CERF sited items, collecting pre-existing only
bau_plants_df = bau_plants_df[bau_plants_df.cerf_sited == 0]

# remove wind facilities
bau_plants_df = bau_plants_df[bau_plants_df.technology_simple != 'Wind']

# reduce to retired plants
bau_plants_retired = bau_plants_df[bau_plants_df.retirement_year <= 2050]
geometry = [Point(xy) for xy in zip(bau_plants_retired['xcoord'], bau_plants_retired['ycoord'])]
bau_non_cerf_retired_gdf = gpd.GeoDataFrame(bau_plants_retired, crs='ESRI:102003', geometry=geometry)
bau_non_cerf_retired_gdf.set_crs('ESRI:102003', inplace=True)

# Pre-existing plants (wind)
wind_coord_df = gpd.read_file(wind_coord_file)
wind_coord_df.set_crs('ESRI:102003', inplace=True)
wind_coord_df = wind_coord_df.rename(columns={'State': 'region_name'})
wind_coord_df['technology_simple'] = 'Wind'

# combine together
bau_retired = pd.concat([bau_cerf_retired_gdf, bau_non_cerf_retired_gdf, wind_coord_df])

# add scenario to missing
bau_retired['scenario'] = 'business_as_usual_ira_ccs_climate'

# reduce to columns of interest
bau_retired = bau_retired[['scenario', 'technology_simple', 'region_name', 'geometry']]

# square buffer points to get project level area
bau_retired.geometry  = bau_retired.geometry.buffer(500, cap_style='square')

New infrastruture development

In [10]:
# read in 1km file to get cerf-sited plants
bau_1km_df = pd.read_csv(bau_1km_path)

# create geometry
geometry = [Point(xy) for xy in zip(bau_1km_df['xcoord'], bau_1km_df['ycoord'])]
bau_cerf_sited_gdf = gpd.GeoDataFrame(bau_1km_df, crs='ESRI:102003', geometry=geometry)
bau_cerf_sited_gdf.set_crs('ESRI:102003', inplace=True)
# square buffer points to get project level area
bau_cerf_sited_gdf.geometry  = bau_cerf_sited_gdf.geometry.buffer(500, cap_style='square')

# dissolve to single geometry
bau_cerf_sited_gdf = bau_cerf_sited_gdf.dissolve()

Take the difference to determine land that had retirement but saw no new infrastructure

In [11]:
# the overlap: retired areas where there was new generation
bau_difference = gpd.clip(bau_retired, bau_cerf_sited_gdf)

# remove the overlap from the retired land
bau_retired_open = gpd.overlay(bau_retired, bau_difference, how='difference')

# get land area
bau_retired_open['value'] = bau_retired_open['geometry'].area

# convert to km2
bau_retired_open['value'] = (bau_retired_open['value'] / 1e6) * -1

# rename columns
bau_retired_open = bau_retired_open.rename(columns={'technology_simple':'Technology', 'region_name': 'State'})

# reduce to columns of interest
bau_retired_open = bau_retired_open[['Technology', 'State', 'scenario', 'value']]

# add variable name
bau_retired_open['var'] = 'km-squared'

# sum by state and technology
bau_retired_open = bau_retired_open.groupby(['Technology', 'State', 'scenario', 'var'], as_index=False).sum()

# west states only
bau_retired_open = bau_retired_open[bau_retired_open.State.isin(west_states)]

#### Combine all data and save to file

In [12]:
hr_df_cap_retired['scenario'] = 'High Renewables'
hr_retired_open['scenario'] = 'High Renewables'
bau_cap_retired['scenario'] = 'Business-as-usual'
bau_retired_open['scenario'] = 'Business-as-usual'

# combine scenario data
output_df = pd.concat([hr_df_cap_retired, hr_retired_open, bau_cap_retired, bau_retired_open])

output_df = output_df.rename(columns={'region_name':'State'})
output_df['State'] = output_df['State'].str.title()
output_df['State'] = output_df['State'].replace("New_Mexico", "New Mexico")

# save to file
output_df.to_csv(retirement_output_path, index=False)